In [1]:
import dill
import json
import string

import numpy as np
import scipy
import pandas as pd

import seaborn as sns

from collections import Counter

In [2]:
### UTIL
    
def clean(text):
    text = text.lower()
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.translate(str.maketrans('', '', string.punctuation))

    return text

In [3]:
processed_data_path = '../data/processed/'

corpus_narr_fn = processed_data_path + 'corpus_narr.pkl'
corpus_else_fn = processed_data_path + 'corpus_else.pkl'
fields_fn = processed_data_path + 'fields.pkl'

wordlists_narr_fn = processed_data_path + 'wordlists_narr.json'
wordlists_else_fn = processed_data_path + 'wordlists_else.json'

In [4]:
corpus_narr = dill.load(open(corpus_narr_fn, 'rb'))
corpus_else = dill.load(open(corpus_else_fn, 'rb'))
fields = dill.load(open(fields_fn, 'rb'))

In [5]:
len(fields)

265

In [6]:
# WORDLISTS

def get_wordlist(corp):
    wl = [clean(w['lemma']) for doc in corp for sen in corp[doc] for w in corp[doc][sen]['spc']]
    return [w for w in wl if w != '']

def get_wordlists(doreco):
    return {lang: get_wordlist(doreco[lang]) for lang in doreco}

In [7]:
wls_narr = get_wordlists(corpus_narr)
wls_else = get_wordlists(corpus_else)

In [8]:
json.dump(wls_narr, open(wordlists_narr_fn, 'w'))
json.dump(wls_else, open(wordlists_else_fn, 'w'))

In [12]:
for lang in wls_narr:
    try:
        print(lang, len(wls_narr[lang]), len(wls_else[lang]))
    except:
        continue

anal1239 15807 1862
apah1238 12499 5248
arap1274 27520 627
bain1259 6702 13965
beja1238 28791 2005
bora1263 40748 4902
cabe1245 2714 4519
dolg1241 24727 2481
even1259 15140 162
goem1240 19987 12429
goro1270 7180 20551
hoch1243 8339 1222
kaka1265 10286 35656
kama1351 56106 938
komn1238 36067 6209
nngg1234 20669 10733
nort2875 2890 11647
orko1234 9538 7726
pnar1238 14564 1561
ruul1235 3600 12966
sadu1234 8013 2559
savo1255 8532 456
sumi1235 438 9277
trin1278 22923 2005
yong1270 13096 6787
yuca1254 2088 6725


In [16]:
a = [0,1,2,3,4,5]
i=3
n=5
j=i+4
a[i:n+1] + a[0:j-n]
a[0:j-n]

[0, 1]

In [9]:
### USAGE FREQUENCY

def sample(a, size):
    n = len(a)
    i = np.random.randint(0, n)
    j = i+size
    
    sample = a[i:j] if j <= n else a[i:n] + a[0:j-n]

    return sample
    
def narr_else_frequencies(field, wl_narr, wl_else, n_samples=100):
    bigger = wl_narr if len(wl_narr) > len(wl_else) else wl_else
    smaller = wl_else if len(wl_narr) > len(wl_else) else wl_narr

    sampled = [frequency(field, sample(bigger, len(smaller))) for i in range(n_samples)]
    #print(np.mean(sampled_freqs), np.std(sampled_freqs))
    
    narr_freq = np.mean(sampled) if wl_narr == bigger else frequency(field, wl_narr)
    else_freq = np.mean(sampled) if wl_else == bigger else frequency(field, wl_else)
        
    return narr_freq, else_freq
        
def frequency(field, wordlist):
    freqs = Counter(wordlist)
    count = sum([freqs[w] for w in field if w in freqs])
    total = len(wordlist)
    return count / total

In [10]:
### FIELD RICHNESS

def h(X):
    p = lambda v: v / np.sum(X)
    return -np.sum([p(x) * np.log(p(x)) for x in X])

def measure_langfield_count(langfield):
    return len(langfield)

def measure_langfield_entropy(langfield):
    return h(list(langfield.values()))

In [11]:
### DATA TABLE

FREQS = pd.read_csv(processed_data_path + 'freqs.csv')

def populate_data(fields, wls_narr, wls_else):
    d = {
        'field': list(),
        'lang': list(),
        'narr_freq': list(),
        'narr_freq_log': list(),
        'else_freq': list(), 
        'else_freq_log': list(),
        'lf_count' : list(),
        'lf_count_log': list(),
        'lf_entropy': list()
    }
    
    i = 0
    for field, langfields in fields:
        for lang in langfields:
            if lang in wls_narr and lang in wls_else:
                
                narr_freq, else_freq = narr_else_frequencies(field, wls_narr[lang], wls_else[lang])
                
                lf_count = measure_langfield_count(langfields[lang])
                lf_entropy = measure_langfield_entropy(langfields[lang])
    
                d['field'].append(field)
                d['lang'].append(lang)
                d['narr_freq'].append(narr_freq)
                d['narr_freq_log'].append(np.log(1 + narr_freq * 1e6))
                d['else_freq'].append(else_freq)
                d['else_freq_log'].append(np.log(1 + else_freq * 1e6))
                d['lf_count'].append(lf_count)
                d['lf_count_log'].append(np.log(lf_count + 1))
                d['lf_entropy'].append(lf_entropy)

        i+=1
        print(i, '/', len(fields))
        
    frame = pd.DataFrame(d)
    
    return frame

In [12]:
df = populate_data(fields, wls_narr, wls_else)
df

1 / 304
2 / 304
3 / 304
4 / 304
5 / 304
6 / 304
7 / 304
8 / 304
9 / 304
10 / 304
11 / 304
12 / 304
13 / 304
14 / 304
15 / 304
16 / 304
17 / 304
18 / 304
19 / 304
20 / 304
21 / 304
22 / 304
23 / 304
24 / 304
25 / 304
26 / 304
27 / 304
28 / 304
29 / 304
30 / 304
31 / 304
32 / 304
33 / 304
34 / 304
35 / 304
36 / 304
37 / 304
38 / 304
39 / 304
40 / 304
41 / 304
42 / 304
43 / 304
44 / 304
45 / 304
46 / 304
47 / 304
48 / 304
49 / 304
50 / 304
51 / 304
52 / 304
53 / 304
54 / 304
55 / 304
56 / 304
57 / 304
58 / 304
59 / 304
60 / 304
61 / 304
62 / 304
63 / 304
64 / 304
65 / 304
66 / 304
67 / 304
68 / 304
69 / 304
70 / 304
71 / 304
72 / 304
73 / 304
74 / 304
75 / 304
76 / 304
77 / 304
78 / 304
79 / 304
80 / 304
81 / 304
82 / 304
83 / 304
84 / 304
85 / 304
86 / 304
87 / 304
88 / 304
89 / 304
90 / 304
91 / 304
92 / 304
93 / 304
94 / 304
95 / 304
96 / 304
97 / 304
98 / 304
99 / 304
100 / 304
101 / 304
102 / 304
103 / 304
104 / 304
105 / 304
106 / 304
107 / 304
108 / 304
109 / 304
110 / 304
111 / 30

,field,lang,narr_freq,narr_freq_log,else_freq,else_freq_log,lf_count,lf_count_log,lf_entropy
0,"[remove, carry, come, arrive, leave, bring, ge...",anal1239,0.019823,9.894637,0.014501,9.582010,0,0.000000,-0.000000
1,"[remove, carry, come, arrive, leave, bring, ge...",apah1238,0.016341,9.701522,0.010671,9.275354,0,0.000000,-0.000000
2,"[remove, carry, come, arrive, leave, bring, ge...",arap1274,0.013301,9.495702,0.014354,9.571858,54,4.007333,3.275359
3,"[remove, carry, come, arrive, leave, bring, ge...",bain1259,0.032677,10.394452,0.026162,10.172114,51,3.951244,2.905421
4,"[remove, carry, come, arrive, leave, bring, ge...",beja1238,0.024369,10.101111,0.017456,9.767517,0,0.000000,-0.000000
...,...,...,...,...,...,...,...,...,...
7899,"[love, pity]",savo1255,0.000000,0.000000,0.000000,0.000000,0,0.000000,-0.000000
7900,"[love, pity]",sumi1235,0.000000,0.000000,0.000297,5.696434,0,0.000000,-0.000000
7901,"[love, pity]",trin1278,0.000474,6.162926,0.000000,0.000000,2,1.098612,0.661563
7902,"[love, pity]",yong1270,0.001404,7.247903,0.000000,0.000000,2,1.098612,0.540204


In [13]:
# filter dataframe
data = df[(df.narr_freq != 0) & (df.else_freq != 0)]
data = data[data.lf_count != 0]
data

,field,lang,narr_freq,narr_freq_log,else_freq,else_freq_log,lf_count,lf_count_log,lf_entropy
2,"[remove, carry, come, arrive, leave, bring, ge...",arap1274,0.013301,9.495702,0.014354,9.571858,54,4.007333,3.275359
3,"[remove, carry, come, arrive, leave, bring, ge...",bain1259,0.032677,10.394452,0.026162,10.172114,51,3.951244,2.905421
5,"[remove, carry, come, arrive, leave, bring, ge...",bora1263,0.023268,10.054880,0.017748,9.784076,98,4.595120,3.679492
7,"[remove, carry, come, arrive, leave, bring, ge...",dolg1241,0.019919,9.899499,0.024587,10.110008,44,3.806662,2.481308
10,"[remove, carry, come, arrive, leave, bring, ge...",goro1270,0.021448,9.973455,0.013338,9.498480,57,4.060443,3.504727
...,...,...,...,...,...,...,...,...,...
7873,"[come, return]",savo1255,0.010943,9.300545,0.013158,9.484853,3,1.386294,0.431363
7876,"[come, return]",yong1270,0.005857,8.675527,0.001915,7.558218,12,2.564949,2.195514
7877,"[come, return]",yuca1254,0.001437,7.270857,0.001375,7.226588,2,1.098612,0.636514
7878,"[love, pity]",anal1239,0.001412,7.253796,0.001611,7.385337,4,1.609438,1.157033


In [15]:
data.to_csv(processed_data_path + 'data.csv')